In [29]:
# Standard library
import os
import re
from pathlib import Path

# Third-party (alphabetical)
import numpy as np
import pandas as pd
import torch
from dotenv import load_dotenv
from openai import OpenAI
from transformers import AutoTokenizer, AutoModel

# Project paths
PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data"

In [30]:

# Reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Paths
PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data"

# Chunking configuration
FIXED_CHUNK_WORDS = 150
FIXED_CHUNK_OVERLAP = 30

# Columns we expect in the clinical notes dataset
REQUIRED_NOTE_COLUMNS = {
    "person_id",
    "admission_id",
    "clinical_note_id",
    "clean_note_text",
    "creation_timestamp",
    "note_subject",
    "note_type",
}

In [31]:
NOTES_PATH =  '../data/raw/clinical_notes.csv'   # update if filename/path differs

notes = pd.read_csv(NOTES_PATH)


print(f"Loaded {len(notes):,} clinical notes")
notes.head()

Loaded 1,602 clinical notes


,ingest_timestamp,clinical_note_id,clean_note_text,creation_timestamp,updt_dt_tm,note_subject,note_type,admission_id,person_id
0,07/01/2026 14:35,17bf845b-88f8-4604-8983-6e74453aada5,Patient Name: Judith Ada Wells\n- Patient ID: ...,07/01/2026 14:05,07/01/2026 14:35,ED Triage,ED,63720303-3c1b-4356-befd-eea5438da62e,28570119-9cdc-4120-98c0-4edb76cf36a3
1,07/01/2026 14:50,e5d9c0a4-299a-425e-abbc-27fabe9cb742,Patient reviewed at 14:20 on 07/01/26 by Nurse...,07/01/2026 14:20,07/01/2026 14:50,ED Triage Follow-Up,ED,63720303-3c1b-4356-befd-eea5438da62e,28570119-9cdc-4120-98c0-4edb76cf36a3
2,07/01/2026 15:15,1a711621-1094-4d0b-9cec-7925438e19cb,"- Patient: Judith Ad a Wells, 39-year-old fema...",07/01/2026 14:45,07/01/2026 15:15,ED CT Head Scan Review,ED,63720303-3c1b-4356-befd-eea5438da62e,28570119-9cdc-4120-98c0-4edb76cf36a3
3,07/01/2026 15:45,bbbb3acb-d58e-414d-a0de-7c29553b5459,"Patient: Judith Ada Wells, 39-yer-old female, ...",07/01/2026 15:15,07/01/2026 15:45,ED Investigations,ED,63720303-3c1b-4356-befd-eea5438da62e,28570119-9cdc-4120-98c0-4edb76cf36a3
4,07/01/2026 17:00,e3ec2bf0-baa8-4287-8698-592f70a4bccf,Patient\nJudith Ada Wells\n\nAge\n39\n\nSex\nF...,07/01/2026 16:30,07/01/2026 17:00,ED Depart Summary,ED Depart Summary,63720303-3c1b-4356-befd-eea5438da62e,28570119-9cdc-4120-98c0-4edb76cf36a3


In [32]:
missing_columns = REQUIRED_NOTE_COLUMNS - set(notes.columns)

if missing_columns:
    raise ValueError(
        f"Missing required columns: {sorted(missing_columns)}"
    )

notes = notes.copy()

notes["creation_timestamp"] = pd.to_datetime(
    notes["creation_timestamp"],
    errors="coerce"
)

notes["clean_note_text"] = (
    notes["clean_note_text"]
    .fillna("")
    .astype(str)
    .str.strip()
)

# Remove rows without usable note text
notes = notes.loc[
    notes["clean_note_text"].ne("")
].reset_index(drop=True)

notes["word_count"] = (
    notes["clean_note_text"]
    .str.split()
    .str.len()
)

print(f"Notes:      {len(notes):,}")
print(f"Patients:   {notes['person_id'].nunique():,}")
print(f"Admissions: {notes['admission_id'].nunique():,}")

Notes:      1,602
Patients:   50
Admissions: 69


In [33]:
def create_whole_note_chunks(notes_df: pd.DataFrame) -> pd.DataFrame:
    """
    Create one retrieval chunk per clinical note.
    """

    chunks = notes_df[
        [
            "person_id",
            "admission_id",
            "clinical_note_id",
            "creation_timestamp",
            "note_subject",
            "note_type",
            "clean_note_text",
        ]
    ].copy()

    chunks = chunks.rename(
        columns={"clean_note_text": "chunk_text"}
    )

    chunks["chunk_index"] = 0

    chunks["chunk_id"] = (
        chunks["clinical_note_id"].astype(str)
        + "_whole_0"
    )

    chunks["chunk_strategy"] = "whole_note"

    return chunks


whole_chunks = create_whole_note_chunks(notes)

print(f"Whole-note chunks: {len(whole_chunks):,}")
whole_chunks.head()

Whole-note chunks: 1,602


,person_id,admission_id,clinical_note_id,creation_timestamp,note_subject,note_type,chunk_text,chunk_index,chunk_id,chunk_strategy
0,28570119-9cdc-4120-98c0-4edb76cf36a3,63720303-3c1b-4356-befd-eea5438da62e,17bf845b-88f8-4604-8983-6e74453aada5,2026-07-01 14:05:00,ED Triage,ED,Patient Name: Judith Ada Wells\n- Patient ID: ...,0,17bf845b-88f8-4604-8983-6e74453aada5_whole_0,whole_note
1,28570119-9cdc-4120-98c0-4edb76cf36a3,63720303-3c1b-4356-befd-eea5438da62e,e5d9c0a4-299a-425e-abbc-27fabe9cb742,2026-07-01 14:20:00,ED Triage Follow-Up,ED,Patient reviewed at 14:20 on 07/01/26 by Nurse...,0,e5d9c0a4-299a-425e-abbc-27fabe9cb742_whole_0,whole_note
2,28570119-9cdc-4120-98c0-4edb76cf36a3,63720303-3c1b-4356-befd-eea5438da62e,1a711621-1094-4d0b-9cec-7925438e19cb,2026-07-01 14:45:00,ED CT Head Scan Review,ED,"- Patient: Judith Ad a Wells, 39-year-old fema...",0,1a711621-1094-4d0b-9cec-7925438e19cb_whole_0,whole_note
3,28570119-9cdc-4120-98c0-4edb76cf36a3,63720303-3c1b-4356-befd-eea5438da62e,bbbb3acb-d58e-414d-a0de-7c29553b5459,2026-07-01 15:15:00,ED Investigations,ED,"Patient: Judith Ada Wells, 39-yer-old female, ...",0,bbbb3acb-d58e-414d-a0de-7c29553b5459_whole_0,whole_note
4,28570119-9cdc-4120-98c0-4edb76cf36a3,63720303-3c1b-4356-befd-eea5438da62e,e3ec2bf0-baa8-4287-8698-592f70a4bccf,2026-07-01 16:30:00,ED Depart Summary,ED Depart Summary,Patient\nJudith Ada Wells\n\nAge\n39\n\nSex\nF...,0,e3ec2bf0-baa8-4287-8698-592f70a4bccf_whole_0,whole_note


In [34]:
def split_fixed_words(
    text: str,
    chunk_size: int = 150,
    overlap: int = 30,
) -> list[str]:
    """
    Split text into fixed-size word chunks with overlap.
    """

    if chunk_size <= 0:
        raise ValueError("chunk_size must be greater than 0")

    if overlap < 0:
        raise ValueError("overlap cannot be negative")

    if overlap >= chunk_size:
        raise ValueError("overlap must be smaller than chunk_size")

    words = text.split()

    if len(words) <= chunk_size:
        return [text]

    chunks = []
    step = chunk_size - overlap

    for start in range(0, len(words), step):
        end = start + chunk_size
        chunk_words = words[start:end]

        if not chunk_words:
            break

        chunks.append(" ".join(chunk_words))

        if end >= len(words):
            break

    return chunks


def create_fixed_word_chunks(
    notes_df: pd.DataFrame,
    chunk_size: int = 150,
    overlap: int = 30,
) -> pd.DataFrame:
    """
    Create fixed-size word chunks from each clinical note.
    """

    records = []

    for row in notes_df.itertuples(index=False):

        text_chunks = split_fixed_words(
            row.clean_note_text,
            chunk_size=chunk_size,
            overlap=overlap,
        )

        for chunk_index, chunk_text in enumerate(text_chunks):

            records.append(
                {
                    "person_id": row.person_id,
                    "admission_id": row.admission_id,
                    "clinical_note_id": row.clinical_note_id,
                    "creation_timestamp": row.creation_timestamp,
                    "note_subject": row.note_subject,
                    "note_type": row.note_type,
                    "chunk_index": chunk_index,
                    "chunk_id": (
                        f"{row.clinical_note_id}"
                        f"_fixed_{chunk_index}"
                    ),
                    "chunk_strategy": "fixed_words",
                    "chunk_text": chunk_text,
                }
            )

    return pd.DataFrame(records)


fixed_chunks = create_fixed_word_chunks(
    notes,
    chunk_size=FIXED_CHUNK_WORDS,
    overlap=FIXED_CHUNK_OVERLAP,
)

print(f"Fixed-word chunks: {len(fixed_chunks):,}")
fixed_chunks.head()

Fixed-word chunks: 2,298


,person_id,admission_id,clinical_note_id,creation_timestamp,note_subject,note_type,chunk_index,chunk_id,chunk_strategy,chunk_text
0,28570119-9cdc-4120-98c0-4edb76cf36a3,63720303-3c1b-4356-befd-eea5438da62e,17bf845b-88f8-4604-8983-6e74453aada5,2026-07-01 14:05:00,ED Triage,ED,0,17bf845b-88f8-4604-8983-6e74453aada5_fixed_0,fixed_words,Patient Name: Judith Ada Wells\n- Patient ID: ...
1,28570119-9cdc-4120-98c0-4edb76cf36a3,63720303-3c1b-4356-befd-eea5438da62e,e5d9c0a4-299a-425e-abbc-27fabe9cb742,2026-07-01 14:20:00,ED Triage Follow-Up,ED,0,e5d9c0a4-299a-425e-abbc-27fabe9cb742_fixed_0,fixed_words,Patient reviewed at 14:20 on 07/01/26 by Nurse...
2,28570119-9cdc-4120-98c0-4edb76cf36a3,63720303-3c1b-4356-befd-eea5438da62e,1a711621-1094-4d0b-9cec-7925438e19cb,2026-07-01 14:45:00,ED CT Head Scan Review,ED,0,1a711621-1094-4d0b-9cec-7925438e19cb_fixed_0,fixed_words,"- Patient: Judith Ad a Wells, 39-year-old fema..."
3,28570119-9cdc-4120-98c0-4edb76cf36a3,63720303-3c1b-4356-befd-eea5438da62e,bbbb3acb-d58e-414d-a0de-7c29553b5459,2026-07-01 15:15:00,ED Investigations,ED,0,bbbb3acb-d58e-414d-a0de-7c29553b5459_fixed_0,fixed_words,"Patient: Judith Ada Wells, 39-yer-old female, ..."
4,28570119-9cdc-4120-98c0-4edb76cf36a3,63720303-3c1b-4356-befd-eea5438da62e,e3ec2bf0-baa8-4287-8698-592f70a4bccf,2026-07-01 16:30:00,ED Depart Summary,ED Depart Summary,0,e3ec2bf0-baa8-4287-8698-592f70a4bccf_fixed_0,fixed_words,Patient Judith Ada Wells Age 39 Sex Female NHS...


In [35]:
SECTION_HEADINGS = [
    "Presenting Complaint",
    "History of Presenting Illness",
    "History of Present Illness",
    "HPI",
    "Review of Systems",
    "Past Medical History",
    "PMH",
    "Medications",
    "Medication",
    "Allergies",
    "Social History",
    "Family History",
    "On Examination",
    "Examination",
    "Observations",
    "Investigations",
    "Test Results",
    "Results",
    "Assessment",
    "Impression",
    "Diagnosis",
    "Treatment",
    "Plan",
]

SECTION_PATTERN = re.compile(
    rf"(?im)^(?:{'|'.join(map(re.escape, SECTION_HEADINGS))})\s*:?\s*$"
)


def is_heading_only(text: str) -> bool:
    """
    Return True if the chunk contains only a recognized section heading
    and no clinical content.
    """
    lines = [
        line.strip()
        for line in text.splitlines()
        if line.strip()
    ]

    if len(lines) != 1:
        return False

    return bool(SECTION_PATTERN.fullmatch(lines[0]))


def split_by_sections(text: str) -> list[str]:
    """
    Split a clinical note using known section headings.

    - Keeps section heading together with its content.
    - Drops empty sections that contain only a heading.
    - Falls back to the whole note when usable section boundaries
      are not found.
    """

    matches = list(SECTION_PATTERN.finditer(text))

    if len(matches) < 2:
        return [text]

    chunks = []

    # Preserve text before first recognized heading
    prefix = text[:matches[0].start()].strip()

    if prefix:
        chunks.append(prefix)

    for index, match in enumerate(matches):

        start = match.start()

        if index + 1 < len(matches):
            end = matches[index + 1].start()
        else:
            end = len(text)

        section = text[start:end].strip()

        if section and not is_heading_only(section):
            chunks.append(section)

    return chunks


def create_section_chunks(
    notes_df: pd.DataFrame,
) -> pd.DataFrame:
    """
    Create section-aware chunks from clinical notes.

    Notes without detectable sections remain whole.
    """

    records = []

    for row in notes_df.itertuples(index=False):

        text_chunks = split_by_sections(
            row.clean_note_text
        )

        for chunk_index, chunk_text in enumerate(text_chunks):

            records.append(
                {
                    "person_id": row.person_id,
                    "admission_id": row.admission_id,
                    "clinical_note_id": row.clinical_note_id,
                    "creation_timestamp": row.creation_timestamp,
                    "note_subject": row.note_subject,
                    "note_type": row.note_type,
                    "chunk_index": chunk_index,
                    "chunk_id": (
                        f"{row.clinical_note_id}"
                        f"_section_{chunk_index}"
                    ),
                    "chunk_strategy": "section",
                    "chunk_text": chunk_text,
                }
            )

    return pd.DataFrame(records)


section_chunks = create_section_chunks(notes)

print(f"Section chunks: {len(section_chunks):,}")
section_chunks.head()

Section chunks: 5,189


,person_id,admission_id,clinical_note_id,creation_timestamp,note_subject,note_type,chunk_index,chunk_id,chunk_strategy,chunk_text
0,28570119-9cdc-4120-98c0-4edb76cf36a3,63720303-3c1b-4356-befd-eea5438da62e,17bf845b-88f8-4604-8983-6e74453aada5,2026-07-01 14:05:00,ED Triage,ED,0,17bf845b-88f8-4604-8983-6e74453aada5_section_0,section,Patient Name: Judith Ada Wells\n- Patient ID: ...
1,28570119-9cdc-4120-98c0-4edb76cf36a3,63720303-3c1b-4356-befd-eea5438da62e,e5d9c0a4-299a-425e-abbc-27fabe9cb742,2026-07-01 14:20:00,ED Triage Follow-Up,ED,0,e5d9c0a4-299a-425e-abbc-27fabe9cb742_section_0,section,Patient reviewed at 14:20 on 07/01/26 by Nurse...
2,28570119-9cdc-4120-98c0-4edb76cf36a3,63720303-3c1b-4356-befd-eea5438da62e,1a711621-1094-4d0b-9cec-7925438e19cb,2026-07-01 14:45:00,ED CT Head Scan Review,ED,0,1a711621-1094-4d0b-9cec-7925438e19cb_section_0,section,"- Patient: Judith Ad a Wells, 39-year-old fema..."
3,28570119-9cdc-4120-98c0-4edb76cf36a3,63720303-3c1b-4356-befd-eea5438da62e,bbbb3acb-d58e-414d-a0de-7c29553b5459,2026-07-01 15:15:00,ED Investigations,ED,0,bbbb3acb-d58e-414d-a0de-7c29553b5459_section_0,section,"Patient: Judith Ada Wells, 39-yer-old female, ..."
4,28570119-9cdc-4120-98c0-4edb76cf36a3,63720303-3c1b-4356-befd-eea5438da62e,e3ec2bf0-baa8-4287-8698-592f70a4bccf,2026-07-01 16:30:00,ED Depart Summary,ED Depart Summary,0,e3ec2bf0-baa8-4287-8698-592f70a4bccf_section_0,section,Patient\nJudith Ada Wells\n\nAge\n39\n\nSex\nF...


In [36]:
chunk_summary = pd.DataFrame(
    {
        "strategy": [
            "Whole note",
            "Fixed words",
            "Section aware",
        ],
        "num_chunks": [
            len(whole_chunks),
            len(fixed_chunks),
            len(section_chunks),
        ],
        "avg_words_per_chunk": [
            whole_chunks["chunk_text"]
            .str.split()
            .str.len()
            .mean(),

            fixed_chunks["chunk_text"]
            .str.split()
            .str.len()
            .mean(),

            section_chunks["chunk_text"]
            .str.split()
            .str.len()
            .mean(),
        ],
    }
)

chunk_summary

,strategy,num_chunks,avg_words_per_chunk
0,Whole note,1602,129.997503
1,Fixed words,2298,99.711053
2,Section aware,5189,40.124880


In [37]:
section_counts = (
    section_chunks
    .groupby("clinical_note_id")
    .size()
)

print(
    "Notes split into multiple sections:",
    (section_counts > 1).sum()
)

print(
    "Notes left whole:",
    (section_counts == 1).sum()
)

print(
    "Percentage split:",
    round(
        (section_counts > 1).mean() * 100,
        2,
    ),
    "%"
)

Notes split into multiple sections: 569
Notes left whole: 1033
Percentage split: 35.52 %


In [38]:
example_note_id = (
    notes
    .sort_values("word_count", ascending=False)
    .iloc[0]["clinical_note_id"]
)

example_note = notes.loc[
    notes["clinical_note_id"] == example_note_id,
    [
        "clinical_note_id",
        "note_subject",
        "word_count",
        "clean_note_text",
    ],
]

example_note


print("WHOLE NOTE")
print("=" * 80)

display(
    whole_chunks.loc[
        whole_chunks["clinical_note_id"] == example_note_id,
        ["chunk_index", "chunk_text"],
    ]
)

print("\nFIXED WORD")
print("=" * 80)

display(
    fixed_chunks.loc[
        fixed_chunks["clinical_note_id"] == example_note_id,
        ["chunk_index", "chunk_text"],
    ]
)

print("\nSECTION")
print("=" * 80)

display(
    section_chunks.loc[
        section_chunks["clinical_note_id"] == example_note_id,
        ["chunk_index", "chunk_text"],
    ]
)

WHOLE NOTE


,chunk_index,chunk_text
93,0,Clerking Doctor\nDr. Kelly Nicola Hayward (SpR...



FIXED WORD


,chunk_index,chunk_text
130,0,Clerking Doctor Dr. Kelly Nicola Hayward (SpR)...
131,1,habit - Renal: No dysuria or haematuria - MSK:...
132,2,family history of neurodegenerative or psychia...
133,3,"air entry bilaterally, no added sounds - Abdom..."
134,4,- Continue monitoring neurological status with...



SECTION


,chunk_index,chunk_text
336,0,Clerking Doctor\nDr. Kelly Nicola Hayward (SpR)
337,1,Presenting Complaint\nAcute confusion followin...
338,2,"Review of Systems\n- CNS: Mild confusion, deni..."
339,3,Past Medical History\n- HTN\n- Mild osteoarthr...
340,4,Medications\n- No regular medications\n- IV pa...
341,5,Allergies\nNone
342,6,Social History\n- Lives alone in a ground-floo...
343,7,"Family History\n- Father: Deceased, history of..."
344,8,"On Examination\n- Alert but mildly confused, G..."
345,9,Observations\nHR 88\nBP 142/86\nRR 16\nTemp 36...


In [39]:
chunk_summary

,strategy,num_chunks,avg_words_per_chunk
0,Whole note,1602,129.997503
1,Fixed words,2298,99.711053
2,Section aware,5189,40.124880


In [40]:
section_counts = (
    section_chunks
    .groupby("clinical_note_id")
    .size()
)

print("Notes split into multiple sections:", (section_counts > 1).sum())
print("Notes left whole:", (section_counts == 1).sum())
print(
    "Percentage split:",
    round((section_counts > 1).mean() * 100, 2),
    "%"
)

Notes split into multiple sections: 569
Notes left whole: 1033
Percentage split: 35.52 %


In [41]:
section_chunks = section_chunks.copy()

section_chunks["chunk_word_count"] = (
    section_chunks["chunk_text"]
    .str.split()
    .str.len()
)

section_chunks["chunk_word_count"].describe(
    percentiles=[0.10, 0.25, 0.50, 0.75, 0.90, 0.95]
)

count    5189.000000
mean       40.124880
std        42.022617
min         1.000000
10%         7.000000
25%        11.000000
50%        29.000000
75%        52.000000
90%        88.000000
95%       130.000000
max       385.000000
Name: chunk_word_count, dtype: float64

In [42]:
for threshold in [5, 10, 20]:
    count = (section_chunks["chunk_word_count"] < threshold).sum()
    percentage = count / len(section_chunks) * 100

    print(
        f"Chunks < {threshold} words: "
        f"{count:,} ({percentage:.1f}%)"
    )

Chunks < 5 words: 321 (6.2%)
Chunks < 10 words: 1,078 (20.8%)
Chunks < 20 words: 2,190 (42.2%)


In [43]:
small_chunks = (
    section_chunks.loc[
        section_chunks["chunk_word_count"] < 10,
        [
            "clinical_note_id",
            "note_subject",
            "chunk_index",
            "chunk_word_count",
            "chunk_text",
        ],
    ]
    .sort_values("chunk_word_count")
)

print(f"Number of chunks <10 words: {len(small_chunks)}")

pd.set_option("display.max_colwidth", None)

small_chunks.head(30)

Number of chunks <10 words: 1078


,clinical_note_id,note_subject,chunk_index,chunk_word_count,chunk_text
4370,a62ff755-7c96-4473-ae8b-f7b468cfb771,Dietary Assessment and Plan,0,1,#NAME?
3246,4bf1b7ee-5bdd-40f6-93b3-f576c7158c66,Dietitian Review,0,1,#NAME?
812,0940fa5e-f355-4cea-8e17-61465ea4ff7c,Physio-led education session,0,1,#NAME?
4834,5f6eacde-a2b9-49dd-ac1d-27ac58b0c2de,Dietary review post-op,0,1,#NAME?
2526,170c8e71-5d7f-4bf7-a74f-41aa2a2d800c,Dietitian review for post-op nutrition,0,1,#NAME?
1682,d8df8be7-3622-4782-857f-8d389ddad694,Dietitian Review,0,1,#NAME?
962,661de17e-a4a7-4549-857a-b78e5ce2decf,Dietitian review for post-op nutrition,0,1,#NAME?
3012,6bc4db9a-6d69-460e-a8d8-1657f6c79dc9,ED Depart Summary,4,2,Allergies\nPollen
1071,13ee5295-939b-4ebf-add9-f8c971f66a42,PMWR Ortho Reg,2,2,Investigations\nNone
3242,10ffbd24-1f2e-4303-8daf-873967db1ea3,Neuro AMWR,2,2,Investigations\nNone


In [44]:
investigation_only = section_chunks.loc[
    section_chunks["chunk_text"].str.strip().eq("Investigations"),
    [
        "clinical_note_id",
        "note_subject",
        "chunk_index",
    ],
]

print(
    f"Standalone 'Investigations' chunks: "
    f"{len(investigation_only)}"
)

investigation_only.head(10)

Standalone 'Investigations' chunks: 0


,clinical_note_id,note_subject,chunk_index


In [45]:
small_chunk_text_counts = (
    section_chunks.loc[
        section_chunks["chunk_word_count"] < 10,
        "chunk_text"
    ]
    .str.strip()
    .value_counts()
    .head(30)
)

small_chunk_text_counts

chunk_text
Investigations\nNone                                                   62
Past Medical History\nNil                                              27
Clinician Leading Ward Round\nDr. Sade Olowoyeye (SpR)                 26
Medications\nNil                                                       25
Medications\nNone                                                      21
Clinician Leading Ward Round\nDr. Marcus John Whitehead (SpR)          20
Allergies\nPollen                                                      19
Clinician Leading Ward Round\nDr. Tao Tang (SpR)                       18
Clinician Leading Ward Round\nDr. Stuart Thomas Payne (SpR)            16
Allergies\nNil                                                         14
Clinician Leading Ward Round\nDr. Brenda Veronica Miles (SpR)          14
Allergies\nNo known drug allergies                                     11
Clinician Leading Ward Round\nDr. Edward Aaron O'Connor (SpR)          10
Allergies\nNone            

In [46]:
section_chunks.loc[
    section_chunks["chunk_text"].str.contains(
        r"#NAME\?",
        na=False,
        regex=True,
    ),
    [
        "clinical_note_id",
        "note_subject",
        "chunk_text",
    ],
].head(20)

,clinical_note_id,note_subject,chunk_text
812,0940fa5e-f355-4cea-8e17-61465ea4ff7c,Physio-led education session,#NAME?
962,661de17e-a4a7-4549-857a-b78e5ce2decf,Dietitian review for post-op nutrition,#NAME?
1682,d8df8be7-3622-4782-857f-8d389ddad694,Dietitian Review,#NAME?
2526,170c8e71-5d7f-4bf7-a74f-41aa2a2d800c,Dietitian review for post-op nutrition,#NAME?
3246,4bf1b7ee-5bdd-40f6-93b3-f576c7158c66,Dietitian Review,#NAME?
4370,a62ff755-7c96-4473-ae8b-f7b468cfb771,Dietary Assessment and Plan,#NAME?
4834,5f6eacde-a2b9-49dd-ac1d-27ac58b0c2de,Dietary review post-op,#NAME?


In [47]:
name_error_ids = section_chunks.loc[
    section_chunks["chunk_text"].str.strip().eq("#NAME?"),
    "clinical_note_id"
].unique()

notes.loc[
    notes["clinical_note_id"].isin(name_error_ids),
    [
        "clinical_note_id",
        "note_subject",
        "clean_note_text",
    ]
]

,clinical_note_id,note_subject,clean_note_text
244,0940fa5e-f355-4cea-8e17-61465ea4ff7c,Physio-led education session,#NAME?
293,661de17e-a4a7-4549-857a-b78e5ce2decf,Dietitian review for post-op nutrition,#NAME?
520,d8df8be7-3622-4782-857f-8d389ddad694,Dietitian Review,#NAME?
787,170c8e71-5d7f-4bf7-a74f-41aa2a2d800c,Dietitian review for post-op nutrition,#NAME?
1014,4bf1b7ee-5bdd-40f6-93b3-f576c7158c66,Dietitian Review,#NAME?
1363,a62ff755-7c96-4473-ae8b-f7b468cfb771,Dietary Assessment and Plan,#NAME?
1498,5f6eacde-a2b9-49dd-ac1d-27ac58b0c2de,Dietary review post-op,#NAME?


In [48]:
INVALID_NOTE_VALUES = {"#NAME?"}

invalid_mask = (
    notes["clean_note_text"]
    .str.strip()
    .isin(INVALID_NOTE_VALUES)
)

print("Invalid notes removed:", invalid_mask.sum())

notes_clean = notes.loc[~invalid_mask].copy()

print("Notes before cleaning:", len(notes))
print("Notes after cleaning:", len(notes_clean))

Invalid notes removed: 7
Notes before cleaning: 1602
Notes after cleaning: 1595


In [49]:
whole_chunks = create_whole_note_chunks(notes_clean)

fixed_chunks = create_fixed_word_chunks(
    notes_clean,
    chunk_size=FIXED_CHUNK_WORDS,
    overlap=FIXED_CHUNK_OVERLAP,
)

section_chunks = create_section_chunks(notes_clean)

## Embedding Models

We compare three embedding approaches:

1. General-purpose: GeminiAI text-embedding-3-small
2. Retrieval-focused: BGE-M3
3. Biomedical retrieval: MedCPT

Each embedding model will be evaluated with the same three chunking strategies:
- Whole-note
- Fixed-size overlapping chunks
- Section-aware chunks

### 1. GeminiAI — text-embedding-3-small

In [50]:
%pip install -q openai


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [51]:
%pip install -q python-dotenv


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [52]:
from google import genai

PROJECT_ROOT = Path.cwd().parent
ENV_PATH = PROJECT_ROOT / ".env"

load_dotenv(ENV_PATH, override=True)

gemini_api_key = os.getenv("GEMINI_API_KEY")

print("Gemini key loaded:", bool(gemini_api_key))

client = genai.Client(api_key=gemini_api_key)

Gemini key loaded: True


In [53]:
# Sanity test: generate one Gemini embedding

test_text = "Patient developed acute confusion following a minor fall."

response = client.models.embed_content(
    model="gemini-embedding-001",
    contents=test_text,
)

test_embedding = response.embeddings[0].values

print("Embedding dimensions:", len(test_embedding))
print("First 10 values:", test_embedding[:10])

Embedding dimensions: 3072
First 10 values: [0.0114165805, -0.014645216, 0.005921604, -0.06154907, 0.00182117, -0.0041579907, -0.014994828, 0.0123460945, 0.0035066789, 0.001972333]


In [54]:
%pip install -q sentence-transformers


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [55]:
%pip install nbqa ruff isort


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [56]:
import os
from pathlib import Path

from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd().parent
ENV_PATH = PROJECT_ROOT / ".env"

load_dotenv(ENV_PATH, override=True)

hf_token = os.getenv("HF_TOKEN")

print("HF token loaded:", bool(hf_token))

HF token loaded: True


In [57]:
from huggingface_hub import login

login(token=hf_token)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [58]:
print("HF token loaded:", bool(hf_token))

HF token loaded: True


In [59]:
login(token=hf_token)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [60]:

PROJECT_ROOT = Path.cwd().parent
ENV_PATH = PROJECT_ROOT / ".env"

print("ENV path:", ENV_PATH)
print("ENV exists:", ENV_PATH.exists())

load_dotenv(ENV_PATH, override=True)

hf_token = os.getenv("HF_TOKEN")

print("HF token loaded:", bool(hf_token))

ENV path: /Users/pallavi_chandanshive/projects/clinical-summarization-eval/.env
ENV exists: True
HF token loaded: True


In [61]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

print(PROJECT_ROOT)

/Users/pallavi_chandanshive/projects/clinical-summarization-eval


In [62]:
from sentence_transformers import SentenceTransformer

BGE_MODEL_PATH = PROJECT_ROOT / "models" / "bge-base-en-v1.5"

bge_model = SentenceTransformer(str(BGE_MODEL_PATH))

test_text = "Patient developed acute confusion following a minor fall."

bge_embedding = bge_model.encode(test_text)

print("Embedding dimensions:", len(bge_embedding))
print("First 10 values:", bge_embedding[:10])

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Embedding dimensions: 768
First 10 values: [-0.00121067 -0.01221776  0.00434872  0.00537568  0.02330717 -0.00891011
  0.05173944  0.01838303 -0.02286186 -0.01189964]


### 3. MedCPT — Biomedical retrieval embedding

In [63]:
%pip install -q transformers torch


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [64]:

MEDCPT_QUERY_PATH = PROJECT_ROOT / "models" / "MedCPT-Query-Encoder"
MEDCPT_ARTICLE_PATH = PROJECT_ROOT / "models" / "MedCPT-Article-Encoder"

print("Query model exists:", MEDCPT_QUERY_PATH.exists())
print("Article model exists:", MEDCPT_ARTICLE_PATH.exists())

Query model exists: True
Article model exists: True


In [65]:
query_tokenizer = AutoTokenizer.from_pretrained(
    MEDCPT_QUERY_PATH,
    local_files_only=True
)

query_model = AutoModel.from_pretrained(
    MEDCPT_QUERY_PATH,
    local_files_only=True
)

article_tokenizer = AutoTokenizer.from_pretrained(
    MEDCPT_ARTICLE_PATH,
    local_files_only=True
)

article_model = AutoModel.from_pretrained(
    MEDCPT_ARTICLE_PATH,
    local_files_only=True
)

query_model.eval()
article_model.eval()

print("MedCPT models loaded successfully")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MedCPT models loaded successfully


In [66]:
test_query = "What caused the patient's acute confusion?"

test_article = """
Patient developed acute confusion following a minor fall.
CT head showed a subdural hygroma. Mild hyponatremia and dehydration
were also documented.
"""

In [67]:
with torch.no_grad():

    query_inputs = query_tokenizer(
        test_query,
        return_tensors="pt",
        truncation=True,
        padding=True,
    )

    query_output = query_model(**query_inputs)
    query_embedding = query_output.last_hidden_state[:, 0, :]


    article_inputs = article_tokenizer(
        test_article,
        return_tensors="pt",
        truncation=True,
        padding=True,
    )

    article_output = article_model(**article_inputs)
    article_embedding = article_output.last_hidden_state[:, 0, :]

In [68]:
print("Query embedding shape:", query_embedding.shape)
print("Article embedding shape:", article_embedding.shape)

print("First 10 query values:")
print(query_embedding[0][:10])

print("\nFirst 10 article values:")
print(article_embedding[0][:10])

Query embedding shape: torch.Size([1, 768])
Article embedding shape: torch.Size([1, 768])
First 10 query values:
tensor([ 0.0749, -0.0908, -0.0609, -0.3313, -0.0904, -0.0067, -0.3357, -0.0689,
        -0.1430, -0.2947])

First 10 article values:
tensor([-0.3596, -0.0933, -0.0052, -0.3221, -0.1385, -0.1546, -0.6965, -0.4492,
        -0.0412,  0.0531])


In [69]:
%whos DataFrame

Variable             Type         Data/Info
-------------------------------------------
chunk_summary        DataFrame    Shape: (3, 3)
example_note         DataFrame    Shape: (1, 4)
fixed_chunks         DataFrame    Shape: (2291, 10)
investigation_only   DataFrame    Shape: (0, 3)
notes                DataFrame    Shape: (1602, 10)
notes_clean          DataFrame    Shape: (1595, 10)
section_chunks       DataFrame    Shape: (5182, 10)
small_chunks         DataFrame    Shape: (1078, 5)
whole_chunks         DataFrame    Shape: (1595, 10)


In [73]:
def embed_gemini_texts(texts):
    response = client.models.embed_content(
        model="gemini-embedding-001",
        contents=texts
    )

    embeddings = [
        embedding.values
        for embedding in response.embeddings
    ]

    return np.array(embeddings)

In [74]:
sample_texts = fixed_chunks["chunk_text"].head(3).tolist()

In [75]:
sample_gemini_embeddings = embed_gemini_texts(sample_texts)

print("Number of embeddings:", len(sample_gemini_embeddings))
print("Embedding shape:", sample_gemini_embeddings.shape)
print("First embedding, first 10 values:")
print(sample_gemini_embeddings[0][:10])

Number of embeddings: 3
Embedding shape: (3, 3072)
First embedding, first 10 values:
[-0.0213872  -0.00441735 -0.02074947 -0.06816971 -0.00032535  0.02437343
  0.01302461  0.00564994 -0.01271442  0.03882335]


In [70]:
sample_texts = fixed_chunks["chunk_text"].head(3).tolist()

sample_bge_embeddings = embed_bge_texts(sample_texts)

print("Number of embeddings:", len(sample_bge_embeddings))
print("Embedding shape:", sample_bge_embeddings.shape)
print("First embedding, first 10 values:")
print(sample_bge_embeddings[0][:10])

Number of embeddings: 3
Embedding shape: (3, 768)
First embedding, first 10 values:
[-0.01965749  0.00617     0.01011455 -0.05260094 -0.01465944  0.02197611
  0.04083324  0.02032509  0.00994131 -0.01185225]


In [71]:
def embed_medcpt_articles(texts):
    inputs = article_tokenizer(
        texts,
        return_tensors="pt",
        padding=True,
        truncation=True
    )

    with torch.no_grad():
        outputs = article_model(**inputs)

    embeddings = outputs.last_hidden_state[:, 0, :]

    return embeddings.cpu().numpy()

In [72]:
sample_medcpt_embeddings = embed_medcpt_articles(sample_texts)

print("Number of embeddings:", len(sample_medcpt_embeddings))
print("Embedding shape:", sample_medcpt_embeddings.shape)
print("First embedding, first 10 values:")
print(sample_medcpt_embeddings[0][:10])

Number of embeddings: 3
Embedding shape: (3, 768)
First embedding, first 10 values:
[-0.12583002  0.03172001 -0.20081128 -0.31064138  0.15516433 -0.20821486
 -0.39055628 -0.27165523 -0.02699574 -0.1275537 ]


In [76]:
# ============================================================
# Configuration-selection experiment: choose one patient
# ============================================================
# We want a patient with multiple admissions and a sufficiently
# rich clinical history so that differences between chunking and
# embedding strategies have a chance to appear.
#
# This table does NOT select the patient automatically.
# It simply summarizes the available longitudinal patients so
# that we can make a deliberate choice.

patient_summary = (
    notes_clean
    .groupby("person_id")
    .agg(
        num_admissions=("admission_id", "nunique"),
        num_notes=("clinical_note_id", "nunique"),
        total_words=("word_count", "sum")
    )
    .reset_index()
)

# Keep only genuinely longitudinal patients:
# patients represented across more than one admission.
longitudinal_patients = (
    patient_summary[
        patient_summary["num_admissions"] > 1
    ]
    .sort_values(
        ["num_admissions", "num_notes", "total_words"],
        ascending=False
    )
    .reset_index(drop=True)
)

print("Number of longitudinal patients:", len(longitudinal_patients))

longitudinal_patients

Number of longitudinal patients: 19


,person_id,num_admissions,num_notes,total_words
0,c6c45c39-cd73-49dd-818d-0a7865fe8a7f,2,90,10268
1,137b8481-4f1d-4b7f-babd-20f7117023ad,2,72,7982
2,69bf7e25-abb2-4dde-857f-f1138d4d0d8a,2,70,11050
3,359014a1-10e6-4bd8-9ba7-513d021c971e,2,68,6272
4,04df53ea-55c1-48d9-84a1-1f15c133b29b,2,66,7250
5,ff8c4724-b7de-4189-bccf-cffddd4d6d44,2,66,6794
6,c50e236f-6b3d-41c8-9e16-7ec343cac820,2,60,7794
7,5e434d78-b2f6-4d88-b327-fff6ee50b901,2,54,6200
8,37b5ce4d-dcfd-4bb7-bee4-d597eb114703,2,52,7050
9,42149ae1-6a3c-471e-a002-cb7263e8bb8c,2,50,6684


In [77]:
# ============================================================
# Inspect the strongest candidate patients
# ============================================================
# We show the patients with the richest longitudinal records.
# We are looking for a patient with:
#   - more than one admission
#   - enough notes to make retrieval meaningful
#   - enough clinical history for chunking differences to matter

longitudinal_patients.head(10)

,person_id,num_admissions,num_notes,total_words
0,c6c45c39-cd73-49dd-818d-0a7865fe8a7f,2,90,10268
1,137b8481-4f1d-4b7f-babd-20f7117023ad,2,72,7982
2,69bf7e25-abb2-4dde-857f-f1138d4d0d8a,2,70,11050
3,359014a1-10e6-4bd8-9ba7-513d021c971e,2,68,6272
4,04df53ea-55c1-48d9-84a1-1f15c133b29b,2,66,7250
5,ff8c4724-b7de-4189-bccf-cffddd4d6d44,2,66,6794
6,c50e236f-6b3d-41c8-9e16-7ec343cac820,2,60,7794
7,5e434d78-b2f6-4d88-b327-fff6ee50b901,2,54,6200
8,37b5ce4d-dcfd-4bb7-bee4-d597eb114703,2,52,7050
9,42149ae1-6a3c-471e-a002-cb7263e8bb8c,2,50,6684


In [78]:
# ============================================================
# Inspect the top 3 candidate longitudinal patients
# ============================================================
# Note count alone does not tell us whether a patient is a good
# configuration-selection case.
#
# We inspect the top candidates to check:
#   1. whether both admissions contain meaningful amounts of data
#   2. whether notes span different note types / clinical events
#   3. whether one admission completely dominates the record
#
# We will then select ONE patient and keep that patient fixed
# across all 9 chunking × embedding configurations.

top_candidate_ids = longitudinal_patients.head(3)["person_id"].tolist()

candidate_overview = (
    notes_clean[
        notes_clean["person_id"].isin(top_candidate_ids)
    ]
    .groupby(["person_id", "admission_id"])
    .agg(
        num_notes=("clinical_note_id", "nunique"),
        total_words=("word_count", "sum"),
        first_note=("creation_timestamp", "min"),
        last_note=("creation_timestamp", "max")
    )
    .reset_index()
    .sort_values(["person_id", "first_note"])
)

candidate_overview

,person_id,admission_id,num_notes,total_words,first_note,last_note
0,137b8481-4f1d-4b7f-babd-20f7117023ad,485363b3-6fcf-4fb9-b005-5aca9e90529a,36,3991,2026-06-01 11:25:00,2026-12-01 10:00:00
1,137b8481-4f1d-4b7f-babd-20f7117023ad,fcc73cbb-cbb3-45ea-a6af-7c428e724f64,36,3991,2026-06-01 11:25:00,2026-12-01 10:00:00
2,69bf7e25-abb2-4dde-857f-f1138d4d0d8a,d8370160-e492-4a63-8751-df815f762726,35,5525,2026-05-01 00:05:00,2026-10-01 09:00:00
3,69bf7e25-abb2-4dde-857f-f1138d4d0d8a,e3a9531a-de58-4142-b5fa-c6b8ccb39d1b,35,5525,2026-05-01 00:05:00,2026-10-01 09:00:00
4,c6c45c39-cd73-49dd-818d-0a7865fe8a7f,7b266c9e-fb25-4209-b856-95bda6915f12,45,5134,2026-03-01 01:35:00,2026-10-01 10:15:00
5,c6c45c39-cd73-49dd-818d-0a7865fe8a7f,8ab5fb09-1638-4054-a031-03291ddf8d07,45,5134,2026-03-01 01:35:00,2026-10-01 10:15:00


In [79]:
# ============================================================
# Check whether the two admission IDs actually contain
# different clinical notes for the same patient.
#
# The previous summary showed identical note counts, word counts,
# and date ranges for both admissions. Before selecting a patient
# for the experiment, we need to determine whether these are
# genuinely separate encounters or duplicated admission mappings.
# ============================================================

check_person_id = "c6c45c39-cd73-49dd-818d-0a7865fe8a7f"

patient_check = (
    notes_clean[
        notes_clean["person_id"] == check_person_id
    ]
    [
        [
            "clinical_note_id",
            "admission_id",
            "creation_timestamp",
            "note_type",
            "note_subject"
        ]
    ]
    .sort_values(["creation_timestamp", "clinical_note_id"])
)

patient_check.head(20)

,clinical_note_id,admission_id,creation_timestamp,note_type,note_subject
1096,85834d83-cab8-440a-b4c4-c8f77e9bc658,8ab5fb09-1638-4054-a031-03291ddf8d07,2026-03-01 01:35:00,ED,ED Triage Assessment
602,f191854f-167f-48e6-8311-84081c6fdb76,7b266c9e-fb25-4209-b856-95bda6915f12,2026-03-01 01:35:00,ED,ED Triage Assessment
1097,932bcfd6-3e9a-4b4a-b885-73b84cc2ac9f,8ab5fb09-1638-4054-a031-03291ddf8d07,2026-03-01 02:15:00,ED,ED Oxygen Therapy Intervention
603,b43f1cce-ade5-40d6-aa30-f28d72389f1d,7b266c9e-fb25-4209-b856-95bda6915f12,2026-03-01 02:15:00,ED,ED Oxygen Therapy Intervention
1098,1fd2d3ad-f926-49d7-8d39-b5aec82a0f09,8ab5fb09-1638-4054-a031-03291ddf8d07,2026-03-01 02:45:00,ED,ED Oxygen Therapy Update
604,7d564f2a-c0f7-41fe-9f2c-64c42c0a0d17,7b266c9e-fb25-4209-b856-95bda6915f12,2026-03-01 02:45:00,ED,ED Oxygen Therapy Update
1099,8bddd5d8-539b-4e59-bf20-05f99a9cff77,8ab5fb09-1638-4054-a031-03291ddf8d07,2026-03-01 03:15:00,ED,ED Review
605,b838732f-1383-4aff-b5d8-49629a0edcea,7b266c9e-fb25-4209-b856-95bda6915f12,2026-03-01 03:15:00,ED,ED Review
606,1891df2e-6957-485e-a785-e0b93a925ecc,7b266c9e-fb25-4209-b856-95bda6915f12,2026-03-01 04:00:00,ED Depart Summary,ED Depart Summary
1100,3910c077-88b6-48eb-b157-285365a86b91,8ab5fb09-1638-4054-a031-03291ddf8d07,2026-03-01 04:00:00,ED Depart Summary,ED Depart Summary


In [80]:
# Check whether the SAME clinical_note_id appears under
# more than one admission_id.

note_admission_counts = (
    patient_check
    .groupby("clinical_note_id")["admission_id"]
    .nunique()
)

print("Unique clinical notes:", patient_check["clinical_note_id"].nunique())

print(
    "Notes linked to more than one admission:",
    (note_admission_counts > 1).sum()
)

print(
    "Total rows:",
    len(patient_check)
)

Unique clinical notes: 90
Notes linked to more than one admission: 0
Total rows: 90


In [81]:
# ============================================================
# Check whether the patient's two admissions contain identical
# clinical TEXT or only follow the same synthetic note structure.
#
# The admissions have matching timestamps, note types and subjects.
# Before using this patient for the configuration-selection
# experiment, we need to know whether their clinical content
# actually differs.
# ============================================================

admission_ids = patient_check["admission_id"].unique()

admission_1 = (
    notes_clean[
        (notes_clean["person_id"] == check_person_id) &
        (notes_clean["admission_id"] == admission_ids[0])
    ]
    .sort_values("creation_timestamp")
    .reset_index(drop=True)
)

admission_2 = (
    notes_clean[
        (notes_clean["person_id"] == check_person_id) &
        (notes_clean["admission_id"] == admission_ids[1])
    ]
    .sort_values("creation_timestamp")
    .reset_index(drop=True)
)

# Compare the actual note text row-by-row.
text_matches = (
    admission_1["clean_note_text"].values
    == admission_2["clean_note_text"].values
)

print("Admission 1 notes:", len(admission_1))
print("Admission 2 notes:", len(admission_2))
print("Exactly identical note texts:", text_matches.sum())
print("Different note texts:", (~text_matches).sum())

Admission 1 notes: 45
Admission 2 notes: 45
Exactly identical note texts: 45
Different note texts: 0


In [82]:
# ============================================================
# Validate the apparent multi-admission patients
# ============================================================
# Some patients appear to have multiple admissions, but the patient
# inspected above had identical clinical-note text duplicated across
# two different admission IDs.
#
# Before selecting a patient for the configuration-selection
# experiment, check whether this duplication pattern occurs across
# ALL patients that appear to have more than one admission.
#
# For each patient, we compare the sets of note texts belonging to
# their admissions. If two admissions contain exactly the same set
# of note texts, they are flagged as identical.
# ============================================================

multi_admission_ids = (
    notes_clean.groupby("person_id")["admission_id"]
    .nunique()
)

multi_admission_ids = multi_admission_ids[
    multi_admission_ids > 1
].index

comparison_results = []

for person_id in multi_admission_ids:

    patient_notes = notes_clean[
        notes_clean["person_id"] == person_id
    ]

    admission_ids = patient_notes["admission_id"].unique()

    # This dataset currently appears to contain two admissions for
    # these patients. Compare their actual clinical text.
    if len(admission_ids) == 2:

        texts_1 = set(
            patient_notes[
                patient_notes["admission_id"] == admission_ids[0]
            ]["clean_note_text"]
        )

        texts_2 = set(
            patient_notes[
                patient_notes["admission_id"] == admission_ids[1]
            ]["clean_note_text"]
        )

        comparison_results.append({
            "person_id": person_id,
            "admission_1_notes": len(texts_1),
            "admission_2_notes": len(texts_2),
            "identical_text_sets": texts_1 == texts_2
        })

admission_duplication_check = pd.DataFrame(comparison_results)

print(
    "Patients checked:",
    len(admission_duplication_check)
)

print(
    "Patients with identical admission text:",
    admission_duplication_check["identical_text_sets"].sum()
)

print(
    "Patients with different admission text:",
    (~admission_duplication_check["identical_text_sets"]).sum()
)

admission_duplication_check

Patients checked: 19
Patients with identical admission text: 19
Patients with different admission text: 0


,person_id,admission_1_notes,admission_2_notes,identical_text_sets
0,04df53ea-55c1-48d9-84a1-1f15c133b29b,33,33,True
1,05192757-942f-460d-b4ff-004ec39cc5ee,23,23,True
2,0c04d9fb-d8d0-4ee0-8fc9-0c3f7f148bdf,19,19,True
3,137b8481-4f1d-4b7f-babd-20f7117023ad,36,36,True
4,1705dd0f-011a-492c-b006-b27e03f2f4ed,14,14,True
5,31f9612b-5a6b-48ea-887b-895772a83b99,22,22,True
6,359014a1-10e6-4bd8-9ba7-513d021c971e,34,34,True
7,37b5ce4d-dcfd-4bb7-bee4-d597eb114703,26,26,True
8,42149ae1-6a3c-471e-a002-cb7263e8bb8c,25,25,True
9,5e434d78-b2f6-4d88-b327-fff6ee50b901,27,27,True


In [83]:
# ============================================================
# Remove duplicated clinical notes across admission IDs
# ============================================================
# Data validation showed that all 19 patients with multiple
# admission IDs contain identical sets of clinical-note text
# across those admissions.
#
# These records have different clinical_note_id/admission_id
# values but duplicate clinical content. Keeping both copies
# would cause the summarization and retrieval experiments to
# process the same clinical evidence twice.
#
# We therefore create a NEW dataframe rather than modifying
# notes_clean, preserving the previous cleaning stage.
# ============================================================

notes_dedup = (
    notes_clean
    .sort_values(["person_id", "creation_timestamp"])
    .drop_duplicates(
        subset=["person_id", "clean_note_text"],
        keep="first"
    )
    .reset_index(drop=True)
)

print("Notes before cross-admission deduplication:", len(notes_clean))
print("Notes after cross-admission deduplication:", len(notes_dedup))
print("Duplicate records removed:", len(notes_clean) - len(notes_dedup))
print("Patients remaining:", notes_dedup["person_id"].nunique())

Notes before cross-admission deduplication: 1595
Notes after cross-admission deduplication: 1103
Duplicate records removed: 492
Patients remaining: 50


In [84]:
# ============================================================
# Check the chunking functions currently available
# ============================================================
# We have cleaned and deduplicated the source notes.
# Before regenerating the three experimental chunk datasets,
# inspect the functions already defined in this notebook so we
# reuse the exact same chunking implementation as before.

%whos function

Variable                   Type        Data/Info
------------------------------------------------
create_fixed_word_chunks   function    <function create_fixed_wo<...>rd_chunks at 0x12842cd50>
create_section_chunks      function    <function create_section_chunks at 0x133e4cf60>
create_whole_note_chunks   function    <function create_whole_no<...>te_chunks at 0x1314ec040>
embed_bge_texts            function    <function embed_bge_texts at 0x1314410c0>
embed_gemini_texts         function    <function embed_gemini_texts at 0x12cb0aa30>
embed_medcpt_articles      function    <function embed_medcpt_articles at 0x12cb085c0>
is_heading_only            function    <function is_heading_only at 0x133e4d0c0>
load_dotenv                function    <function load_dotenv at 0x12c8cc880>
login                      function    <function login at 0x12df75f30>
split_by_sections          function    <function split_by_sections at 0x133e4d170>
split_fixed_words          function    <function split_fixed_

In [85]:
# ============================================================
# Regenerate all three chunking strategies after deduplication
# ============================================================
# IMPORTANT:
# The previous whole_chunks, fixed_chunks, and section_chunks
# were generated from notes_clean, which still contained 492
# duplicate clinical-note records.
#
# We now regenerate every chunking strategy from notes_dedup so
# that all later embedding/configuration experiments use only
# unique clinical evidence.
# ============================================================

whole_chunks = create_whole_note_chunks(notes_dedup)

fixed_chunks = create_fixed_word_chunks(notes_dedup)

section_chunks = create_section_chunks(notes_dedup)


# ------------------------------------------------------------
# Confirm the new chunk counts
# ------------------------------------------------------------

print("Unique source notes:", len(notes_dedup))
print("Whole-note chunks:", len(whole_chunks))
print("Fixed-size chunks:", len(fixed_chunks))
print("Section-aware chunks:", len(section_chunks))

Unique source notes: 1103
Whole-note chunks: 1103
Fixed-size chunks: 1589
Section-aware chunks: 3620


In [86]:
# ============================================================
# Select candidates for the 3 × 3 configuration experiment
# ============================================================
# After removing duplicate clinical content across admission IDs,
# admission count is no longer used to select the test patient.
#
# Instead, we rank patients by the richness of their UNIQUE
# chronological clinical record:
#   - number of unique notes
#   - total amount of clinical text
#   - time span covered by the notes
#
# We will inspect the strongest candidates and select ONE patient
# for all 9 chunking × embedding configurations.
# ============================================================

patient_candidates = (
    notes_dedup
    .groupby("person_id")
    .agg(
        num_notes=("clinical_note_id", "nunique"),
        total_words=("word_count", "sum"),
        first_note=("creation_timestamp", "min"),
        last_note=("creation_timestamp", "max")
    )
    .reset_index()
)

# Calculate how much chronological time each patient's record spans.
patient_candidates["span_days"] = (
    patient_candidates["last_note"] -
    patient_candidates["first_note"]
).dt.days

# Show patients with the richest unique records first.
patient_candidates = (
    patient_candidates
    .sort_values(
        ["num_notes", "total_words", "span_days"],
        ascending=False
    )
    .reset_index(drop=True)
)

patient_candidates.head(10)

,person_id,num_notes,total_words,first_note,last_note,span_days
0,c6c45c39-cd73-49dd-818d-0a7865fe8a7f,45,5134,2026-03-01 01:35:00,2026-10-01 10:15:00,214
1,136c7916-4f9b-4e5c-bf01-77e9d2c681a2,37,4278,2026-02-01 21:15:00,2026-09-01 11:00:00,211
2,137b8481-4f1d-4b7f-babd-20f7117023ad,36,3991,2026-06-01 11:25:00,2026-12-01 10:00:00,182
3,69bf7e25-abb2-4dde-857f-f1138d4d0d8a,35,5525,2026-05-01 00:05:00,2026-10-01 09:00:00,153
4,a9827c1c-fb54-4e5b-8bc6-d3099869e671,35,3926,2026-01-01 16:00:00,2026-08-01 10:00:00,211
5,6e93f9d9-213d-4f2c-a1f0-f475dacef554,34,3867,2026-02-01 09:00:00,2026-11-01 09:30:00,273
6,359014a1-10e6-4bd8-9ba7-513d021c971e,34,3136,2026-07-01 00:05:00,2026-12-01 09:00:00,153
7,04df53ea-55c1-48d9-84a1-1f15c133b29b,33,3625,2026-01-01 07:30:00,2026-08-01 09:00:00,212
8,ff8c4724-b7de-4189-bccf-cffddd4d6d44,33,3397,2026-01-01 12:00:00,2026-06-01 10:00:00,150
9,51f15281-8840-4fd0-92de-89188ab8d736,32,3611,2026-01-01 03:40:00,2026-05-01 16:00:00,120


In [87]:
# ============================================================
# Inspect the selected candidate patient's clinical record
# ============================================================
# Candidate was selected because it has the richest unique
# longitudinal record after deduplication:
#   - 45 unique clinical notes
#   - 5,134 total words
#   - 214-day chronological span
#
# Before locking this patient for the 3 × 3 configuration
# experiment, inspect the note sequence to confirm that the
# record contains meaningful clinical progression over time.
# ============================================================

SELECTED_PERSON_ID = "c6c45c39-cd73-49dd-818d-0a7865fe8a7f"

selected_patient_notes = (
    notes_dedup[
        notes_dedup["person_id"] == SELECTED_PERSON_ID
    ]
    .sort_values("creation_timestamp")
    .reset_index(drop=True)
)

print("Number of notes:", len(selected_patient_notes))
print("Total words:", selected_patient_notes["word_count"].sum())
print(
    "Date range:",
    selected_patient_notes["creation_timestamp"].min(),
    "to",
    selected_patient_notes["creation_timestamp"].max()
)

selected_patient_notes[
    [
        "creation_timestamp",
        "note_type",
        "note_subject",
        "word_count"
    ]
]

Number of notes: 45
Total words: 5134
Date range: 2026-03-01 01:35:00 to 2026-10-01 10:15:00


,creation_timestamp,note_type,note_subject,word_count
0,2026-03-01 01:35:00,ED,ED Triage Assessment,116
1,2026-03-01 02:15:00,ED,ED Oxygen Therapy Intervention,86
2,2026-03-01 02:45:00,ED,ED Oxygen Therapy Update,87
3,2026-03-01 03:15:00,ED,ED Review,94
4,2026-03-01 04:00:00,ED Depart Summary,ED Depart Summary,396
5,2026-03-01 04:30:00,Respiratory Inpatients,Medical Clerking,262
6,2026-03-01 09:00:00,Respiratory Medicine Inpatients,PTWR,169
7,2026-03-01 11:00:00,Physiotherapy Documentation,Diaphragmatic Breathing Exercises,68
8,2026-03-01 13:00:00,Respiratory Inpatients,GP update post-CTPA,28
9,2026-03-01 15:00:00,Physiotherapy Documentation,Mobility Aid and Pacing Education,118


In [88]:
SELECTED_PERSON_ID = "c6c45c39-cd73-49dd-818d-0a7865fe8a7f"

In [89]:
# ============================================================
# Prepare the selected patient's three chunk representations
# ============================================================
# The same patient's clinical record will be used for every
# configuration in the 3 × 3 experiment.
#
# The ONLY differences between configurations will be:
#   1. chunking strategy:
#        - whole-note
#        - fixed-size (150 words, 30-word overlap)
#        - section-aware
#
#   2. embedding approach:
#        - Gemini
#        - BGE-base-en-v1.5
#        - MedCPT
#
# Keeping the patient fixed allows us to compare the nine
# configurations on exactly the same underlying clinical record.
# ============================================================

patient_whole_chunks = (
    whole_chunks[
        whole_chunks["person_id"] == SELECTED_PERSON_ID
    ]
    .sort_values("creation_timestamp")
    .reset_index(drop=True)
)

patient_fixed_chunks = (
    fixed_chunks[
        fixed_chunks["person_id"] == SELECTED_PERSON_ID
    ]
    .sort_values("creation_timestamp")
    .reset_index(drop=True)
)

patient_section_chunks = (
    section_chunks[
        section_chunks["person_id"] == SELECTED_PERSON_ID
    ]
    .sort_values("creation_timestamp")
    .reset_index(drop=True)
)


# Confirm how many searchable chunks each strategy creates
# from exactly the same 45-note patient record.

print("Original unique notes:", len(selected_patient_notes))
print("Whole-note chunks:", len(patient_whole_chunks))
print("Fixed-size chunks:", len(patient_fixed_chunks))
print("Section-aware chunks:", len(patient_section_chunks))

Original unique notes: 45
Whole-note chunks: 45
Fixed-size chunks: 56
Section-aware chunks: 148


In [90]:
# ============================================================
# Single retrieval query for configuration selection
# ============================================================
# We use ONE summarization-oriented retrieval query for all nine
# chunking × embedding configurations.
#
# The query represents the information needed by the downstream
# longitudinal summarization task. Its wording remains identical
# across every configuration so that query formulation is held
# constant during the comparison.
# ============================================================

RETRIEVAL_QUERY = (
    "Retrieve the clinically relevant information needed to produce a "
    "comprehensive longitudinal summary of this patient's clinical history, "
    "including major diagnoses, treatments, investigations, clinical "
    "progression, and outcomes."
)

print(RETRIEVAL_QUERY)

Retrieve the clinically relevant information needed to produce a comprehensive longitudinal summary of this patient's clinical history, including major diagnoses, treatments, investigations, clinical progression, and outcomes.


In [91]:
# ============================================================
# Configuration 1: Whole-note chunks + Gemini embeddings
# ============================================================
# This is the first of our 9 chunking × embedding configurations.
#
# We embed:
#   1. the single fixed retrieval query
#   2. all 45 whole-note chunks for the selected patient
#
# Both query and chunks must be embedded using the SAME embedding
# model so that their vectors exist in the same embedding space.
#
# We are NOT generating the clinical summary yet.
# This step only creates the numerical representations needed
# to compare the query with the patient's whole-note chunks.
# ============================================================

# Embed the single fixed retrieval query.
gemini_query_embedding = embed_gemini_texts(
    [RETRIEVAL_QUERY]
)

# Embed all whole-note chunks belonging to the selected patient.
gemini_whole_embeddings = embed_gemini_texts(
    patient_whole_chunks["chunk_text"].tolist()
)

# Check that the expected number of embeddings was produced.
print("Query embedding shape:", gemini_query_embedding.shape)
print("Whole-note embedding shape:", gemini_whole_embeddings.shape)

Query embedding shape: (1, 3072)
Whole-note embedding shape: (45, 3072)


In [92]:
# ============================================================
# Configuration 1: Calculate query-to-chunk similarity
# Whole-note chunks + Gemini embeddings
# ============================================================
# We now compare the ONE retrieval-query embedding against
# each of the 45 whole-note embeddings.
#
# Cosine similarity measures how closely the direction of two
# vectors aligns in the embedding space.
#
# Higher similarity = the chunk is more semantically relevant
# to our longitudinal-summarization retrieval query.
#
# IMPORTANT:
# We are only RANKING the chunks here.
# We have NOT yet decided how many chunks will be retrieved.
# ============================================================

from sklearn.metrics.pairwise import cosine_similarity

# Compare:
#   1 query embedding
#       against
#   45 whole-note embeddings
#
# Result initially has shape (1, 45), so [0] converts it
# into a simple array containing one score per chunk.
gemini_whole_scores = cosine_similarity(
    gemini_query_embedding,
    gemini_whole_embeddings
)[0]

# Create a copy so the original patient chunk dataframe
# remains unchanged.
gemini_whole_results = patient_whole_chunks.copy()

# Attach each chunk's similarity score.
gemini_whole_results["similarity_score"] = gemini_whole_scores

# Rank chunks from most similar to least similar.
gemini_whole_ranked = (
    gemini_whole_results
    .sort_values("similarity_score", ascending=False)
    .reset_index(drop=True)
)

print("Number of ranked chunks:", len(gemini_whole_ranked))

gemini_whole_ranked[
    [
        "creation_timestamp",
        "note_type",
        "note_subject",
        "similarity_score",
        "chunk_text"
    ]
].head(10)

Number of ranked chunks: 45


,creation_timestamp,note_type,note_subject,similarity_score,chunk_text
0,2026-07-01 10:00:00,Physiotherapy Documentation,Physiotherapy: Breathing Exercises and Light Mobilisation,0.646075,"Patient: Abena Bonsu, 43 years old, admitted with progressive breathlessness due to chronic thromboembolic pulmonary HTN. Past medical history includes HTN and asthma. Allergic to nuts. Session conducted on 07/01/26 at 10:00 by Therapist Joan Winifred Shaw (Physical). \n\nThe session focused on light mobilisation and breathing exercises, including diaphragmatic breathing techniques to improve oxygenation and manage e xertional breathlessnes. The patient ambulated 10 metres using a walking frame with minimal assistance. Oxygen saturations remained above 92% on low-flow oxygen throughout the session. The patient tolerated the exercises well.\n\nPlan: Gradually increase mobilisation distance in future sessions. No changes to medication.\nTherapist Joan Winifred Shaw (Physical)"
1,2026-06-01 15:00:00,Physiotherapy Documentation,Physio: Breathing & Mobility,0.645707,"Patient reviewed at bedside on 2026-01-06 at 15:00 by Therapist Joan Winifred Shaw. Phyiso session focused on breathing exercises and light mobility. Diaphragmatic breathing techniques were practised to enhance O2 exchange. Patient completed two short walks along the ward corridor with rest breaks as required. O2 sats remained above 92% throughout the session, and the patient tolerated the exercises well without significant SOB. Plan to progressively increase walking distance over the next two days to build endurance and assess readiness for discharge.\nTherapist Joan Winifred Shaw (Physical)"
2,2026-03-01 04:00:00,ED Depart Summary,ED Depart Summary,0.645647,"Patient\nAbena Bonsu\n\nAge\n43\n\nSex\nFemale\n\nNHS No.:\n395107142\n\nDate/Time\n03/01/2604:00\n\nSeen By\nDr. Brenda Veronica Miles, ED Consultant\n\nPresenting Complaint\nProgressive breathlessness\n\nHistory of Presenting Complaint\n- Progressive worsening of SOB over the last 2 weks\n- SOB worse on exertion\n- No associated CP, haemoptysis, fever, or syncope reported\n- Describes a sensation of tightness in the chest but denies palpitations or wheezing\n\nPast Medical History\n- HTN\n- Asthma\n\nMedications\n- None reported by the patient at the time of admjssion\n\nAllergies\n- Nuts\n\nSocial History\n- Lives alone\n- Non-smoker\n- No alcohol consumption reported\n- Works as a school teacher\n\nFamily History\n- Father: HTN\n- Mother: Asthma\n\nSystems Review\n- Respiratory: Progressive exertional dyspnoea. No haemoptysis, wheezing, or significant cough.\n- Cardiovascular: No palpitations, chest pain, or syncope.\n- Gastrointestinal: No abdominal pain, nausea, or vomitimg.\n- Neurological: No dizziness, confusion, or focal neuro symptoms.\n- General: No fever or weight loss.\n\nOn Examination\n- Appears mildly distressed with laboured breathing\n- Cardiovascular: Elevated JVP. No peripheral oedema noted. Heart sounds normal, no murmurs.\n- Respiratory: Bilateral basal creps heard on aauscultation. No wheeze.\n- Abdomen: Mildly tenddr hepatomegaly. No ascites.\n- Peripheral vascular: No evidence of DVT. No calf tenderness or swelling.\n\nObservations\nBP 92/64 mmHg\nHR 112 bpm\nRR 28 breaths/min\nTemp 36.8Â°C\nSPO2 90% on 2L NC O2\n\nInvestigations\n- Chest X-ray: Bilateral pulmonary congestion, no focal consolidation or pneumothorax. - CT pulmonary angiogram and echocardiogram planed to confirm suspected chronic thromboembolic pulmonary HTN (pending results).\n\nTest Results\n- ABG: PaO2 8.5 kPa, PaCO2 3.8 kPa, pH 7.46, HCO3- 24 mmol/L (compensated rdspiratory alkalosis, mild hypoxia)\n- NT-proBNP: 4,500 pg/mL (elevated, subsequent result after earlier 3,200 pg/mL reading)\n- Blood tests: D-dimer elevated, other results pending.\n\nImpression\nSuspected chronic thromboembolic pulmonary HTN causing progressive breathlessness and cardiac strain.\n\nReferral\nAccepted by Respiratory HDU under Dr. Elizabeth Kathryn Singh. P